# Strukturalne dane wyjściowe

W tym notebooku zobaczycie, jak wymuszać na lokalnym modelu LLM odpowiedzi w przewidywalnym, maszynowo-przetwarzalnym formacie.

Będziemy pracować z kilkoma kluczowymi elementami:

- **Ollama** — uruchomimy model LLM lokalnie, bez korzystania z API chmurowego. Dzięki temu zobaczycie, że structured generation można realizować na własnym modelu, np. `deepseek-r1:7b`, `mistral:7b` albo `qwen2.5:7b`.

- **Pydantic** — zdefiniujemy kontrakt danych. Model `TicketAnalysis` będzie opisywał, jakie pola mają pojawić się w odpowiedzi, jakie typy są dozwolone oraz jakie wartości są legalne, np. `priority` jako enum.

- **JSON Schema** — wygenerujemy schemat na podstawie modelu Pydantic i przekażemy go do Ollama przez `format=schema`. To będzie główny mechanizm wymuszania struktury odpowiedzi.

- **Opcje deterministyczne** — ustawimy parametry generowania, takie jak:

  ```python
  temperature = 0
  top_k = 1
  top_p = 1
  seed = 42
  ```

  Te parametry ograniczają losowość modelu, ale nie gwarantują pełnego determinizmu semantycznego.

* **Free-form vs structured generation** — porównamy zwykłą odpowiedź tekstową z odpowiedzią strukturalną. Zobaczycie, że free-form generation jest niestabilne, trudne do parsowania i podatne na halucynacje, a JSON Schema pozwala uzyskać stabilny format danych.

* **Walidacja syntaktyczna** — sprawdzimy, czy odpowiedź modelu jest poprawnym JSON-em i czy pasuje do modelu Pydantic.

* **Walidacja semantyczna** — sprawdzimy, czy pole `source_quote` rzeczywiście pochodzi z tekstu źródłowego, zamiast być parafrazą albo halucynacją.

* **Repair / retry loop** — jeśli model zwróci niepoprawny wynik, pokażemy, jak naprawić odpowiedź albo ponowić generację z informacją o błędzie.

* **Kanonizacja JSON** — zamienimy obiekt na stabilną, uporządkowaną reprezentację JSON. Jest to przydatne wtedy, gdy chcemy porównywać wyniki między uruchomieniami.

* **Hash / test regresji** — pokażemy, jak zamienić wynik na hash i używać go do testowania stabilności całego pipeline’u.

Najważniejsza idea tego notebooka:

```text
Prompt nie jest kontraktem.
Schemat + walidacja + retry to kontrakt.
```

Nie będziemy więc jedynie „ładnie prosić” modelu o poprawną odpowiedź. Zbudujemy pipeline, który ogranicza model, sprawdza jego wynik i naprawia go, jeśli będzie taka potrzeba.

## Instalacja zależności

Na początku przygotujemy środowisko do pracy z lokalnym modelem LLM.

Zainstalujemy:

- `zstd` — narzędzie potrzebne podczas instalacji i obsługi komponentów systemowych,
- **Ollama** — runtime do lokalnego uruchamiania modeli językowych,
- `ollama` — klient Pythona do komunikacji z lokalnym serwerem Ollama,
- `pydantic` — bibliotekę do definiowania i walidowania kontraktów danych,
- `rich` — bibliotekę do czytelnego wyświetlania wyników,
- `jsonschema` — bibliotekę do dodatkowej walidacji odpowiedzi JSON.

Po wykonaniu tej komórki będziemy mogli uruchomić lokalny serwer Ollama i komunikować się z modelem bez używania zewnętrznego API.

In [1]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama pydantic rich jsonschema


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (337 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently i

## Uruchomienie serwera Ollama

Ollama działa jako lokalny serwer dostępny domyślnie pod adresem `127.0.0.1:11434`.

W tej komórce sprawdzimy, czy serwer już działa. Jeśli tak, nie uruchamiamy go ponownie. Jeśli nie, startujemy proces `ollama serve` i dajemy mu kilka sekund na inicjalizację.

Dzięki temu kolejne komórki będą mogły komunikować się z lokalnym modelem przez klienta Pythona `ollama`.

In [2]:
import subprocess
import time
import socket


def ollama_is_running(host="127.0.0.1", port=11434):
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False


if ollama_is_running():
    print("Ollama server already running")
    process = None
else:
    process = subprocess.Popen(["ollama", "serve"])
    time.sleep(5)
    print("Ollama server started")


Ollama server started


## Wybór i pobranie modelu

W tej komórce wybieramy model, którego będziemy używać w dalszej części notebooka.

Domyślnie korzystamy z `deepseek-r1:7b`, ale możecie wybrać inny model ale pamiętając o ograniczeniach takich jak VRAM. Po wyborze model zostanie pobrany lokalnie przez Ollama.

W kolejnych komórkach zmienna `model` będzie używana przy każdym wywołaniu LLM-a.

In [3]:
model = "deepseek-r1:7b"
!ollama pull {model}
print("Selected model:", model)



Selected model: deepseek-r1:7b


## Konfiguracja bibliotek NVIDIA

W środowiskach takich jak Google Colab biblioteki NVIDIA mogą znajdować się w katalogu `/usr/lib64-nvidia`. W tej komórce ustawiamy zmienną środowiskową `LD_LIBRARY_PATH`, aby Ollama mogła poprawnie znaleźć biblioteki potrzebne do pracy z GPU.

In [4]:
import os

os.environ.update({"LD_LIBRARY_PATH": "/usr/lib64-nvidia"})
print("LD_LIBRARY_PATH:", os.environ["LD_LIBRARY_PATH"])


LD_LIBRARY_PATH: /usr/lib64-nvidia


## Funkcje pomocnicze

W tej komórce przygotowujemy wspólne funkcje i ustawienia, z których będziemy korzystać w dalszej części notebooka. Najważniejsze elementy:

- `DETERMINISTIC_OPTIONS` — ustawienia ograniczające losowość modelu. Będziemy ich używać wtedy, gdy zależy nam na możliwie stabilnych wynikach.
- `STOCHASTIC_OPTIONS` — ustawienia bardziej losowe. Przydadzą się do pokazania, że zwykłe generowanie tekstu może dawać różne odpowiedzi.
- `call_llm(...)` — jedna wspólna funkcja do wywoływania lokalnego modelu przez Ollama.
- `strip_thinking(...)` — usuwa blok `<think>...</think>`, który mogą zwracać modele reasoning, np. DeepSeek R1.
- `canonical_json(...)` — zamienia wynik na stabilny JSON z uporządkowanymi kluczami.
- `sha256_short(...)` — tworzy krótki hash wyniku, który wykorzystamy później do prostego testu regresji.

In [5]:
import json
import re
import hashlib
from typing import Any, Literal

import ollama
from pydantic import BaseModel, Field, ConfigDict, ValidationError
from rich import print


# Ustawienia możliwie deterministyczne.
# temperature=0 ogranicza losowość,
# top_k=1 wybiera tylko najbardziej prawdopodobny token,
# seed ustawia ziarno losowości po stronie runtime,
# num_predict ogranicza maksymalną długość odpowiedzi.
DETERMINISTIC_OPTIONS = {
    "temperature": 0,
    "seed": 42,
    "top_k": 1,
    "top_p": 1,
    "repeat_penalty": 1.0,
    "num_predict": 2048,
    "num_ctx": 4096,
}


# Ustawienia bardziej losowe.
# Użyjemy ich do pokazania, że zwykłe generowanie tekstu
# może dawać różne odpowiedzi przy kolejnych uruchomieniach.
STOCHASTIC_OPTIONS = {
    "temperature": 0.6,
    "top_k": 40,
    "top_p": 0.9,
    "num_predict": 2048,
    "num_ctx": 4096,
}


def get_message_content(response: Any) -> str:
    """
    Pobiera tekst odpowiedzi z obiektu zwracanego przez ollama.chat(...).

    Różne wersje klienta Ollama mogą zwracać odpowiedź jako obiekt
    z atrybutem response.message.content albo jako słownik.
    Ta funkcja obsługuje oba przypadki.
    """
    try:
        return response.message.content
    except AttributeError:
        return response["message"]["content"]


def strip_thinking(text: str) -> str:
    """
    Usuwa blok <think>...</think> z odpowiedzi modelu.

    Modele reasoning, np. DeepSeek R1, mogą zwracać ukryty tok rozumowania
    w takim bloku. W tym kursie interesuje nas finalna odpowiedź,
    dlatego ten fragment usuwamy.
    """
    return re.sub(r"<think>.*?</think>", "", str(text), flags=re.DOTALL).strip()


def call_llm(messages, *, output_format=None, options=None, think=False, debug=False) -> str:
    """
    Wywołuje lokalny model LLM przez Ollama.

    Parametry:
    - messages: lista wiadomości w formacie chat, np. role=user/system,
    - output_format: opcjonalny format odpowiedzi, np. JSON Schema,
    - options: parametry generowania, np. temperature, top_k, seed,
    - think: czy pozwolić modelowi reasoning na generowanie bloku myślenia,
    - debug: czy wypisać surową odpowiedź modelu.

    Funkcja zwraca tekst odpowiedzi po usunięciu bloku <think>.
    """
    payload = {
        "model": model,
        "messages": messages,
        "options": options or DETERMINISTIC_OPTIONS,
        "stream": False,
        "think": think,
    }

    # Jeśli podamy JSON Schema, Ollama użyje go jako formatu wyjścia.
    # To jest główny mechanizm structured generation w tym notebooku.
    if output_format is not None:
        payload["format"] = output_format

    try:
        response = ollama.chat(**payload)
    except TypeError:
        # Starsze wersje klienta Ollama mogą nie obsługiwać parametru think.
        # Wtedy usuwamy go z payloadu i ponawiamy wywołanie.
        payload.pop("think", None)
        response = ollama.chat(**payload)

    content = get_message_content(response)

    if debug:
        print("RAW CONTENT:")
        print(repr(content[:1000]))

    cleaned = strip_thinking(content)

    # Jeśli model zwrócił tylko blok <think>, po czyszczeniu wynik byłby pusty.
    # W takim przypadku zwracamy surową treść, żeby łatwiej zdiagnozować problem.
    if not cleaned and str(content).strip():
        return str(content).strip()

    return cleaned


def canonical_json(value: Any) -> str:
    """
    Zamienia obiekt na stabilną reprezentację JSON.

    Stabilna reprezentacja oznacza:
    - uporządkowane klucze,
    - brak zbędnych spacji,
    - zachowanie polskich znaków.

    Dzięki temu ten sam wynik logiczny daje ten sam tekst JSON.
    """
    if isinstance(value, BaseModel):
        value = value.model_dump()

    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def sha256_short(text: str) -> str:
    """
    Tworzy krótki hash SHA-256.

    Wykorzystamy go później do prostego testu regresji,
    czyli sprawdzenia, czy wynik pipeline'u zmienia się między uruchomieniami.
    """
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]

# LLM: Structured Generation

Celem tego notebooka jest przejście przez praktyczny wariant **structured generation** na lokalnym modelu językowym.

Nie chodzi o samo napisanie w promptcie: „odpowiedz JSON-em”. Chodzi o zbudowanie przepływu, w którym odpowiedź modelu można potraktować jak dane:

- definiujemy kontrakt,
- przekazujemy JSON Schema do lokalnego modelu,
- walidujemy wynik po stronie aplikacji,
- poprawiamy błędy,
- sprawdzamy powtarzalność.

Wszystkie przykłady korzystają z lokalnego modelu uruchomionego przez **Ollama**.


## Dlaczego zwykły prompt nie wystarcza?

LLM jest modelem generującym tekst. Jeżeli poprosimy go o analizę zgłoszenia, może zwrócić odpowiedź w wielu poprawnych dla człowieka formach:

```text
Priorytet: wysoki
```

```json
{"priority": "high"}
```

```text
Moim zdaniem sprawa jest pilna.
```

Dla człowieka to podobne odpowiedzi. Dla programu są to trzy różne formaty.

Dlatego potrzebujemy **structured generation**, czyli generowania odpowiedzi w formacie, który jest stabilny i możliwy do automatycznego przetwarzania.


## Determinizm w LLM-ach

W klasycznym programowaniu deterministyczna funkcja oznacza, że dla tego samego wejścia zawsze dostajemy ten sam wynik.

W LLM-ach jest trudniej. Na wynik wpływa między innymi:

- model,
- kwantyzacja,
- seed,
- temperatura,
- `top_k`,
- `top_p`,
- długość odpowiedzi,
- backend CPU/GPU,
- prompt template używany przez runtime.

W praktyce nie mówimy więc o absolutnym determinizmie, tylko o **kontrolowanej powtarzalności**. A tą powtarzalność postaramy się uzyskać poprzez:

- `temperature=0`,
- `top_k=1`,
- stały `seed`,
- krótki kontrakt JSON,
- JSON Schema,
- walidację po stronie aplikacji.


### Sprawdzenie połączenia z serwerem Ollama

In [6]:
print("Available local models:")
print(ollama.list())


Available local models:

ListResponse(
    models=[
        Model(
            model='deepseek-r1:7b',
            modified_at=datetime.datetime(2026, 4, 28, 6, 12, 32, 849320, tzinfo=TzInfo(0)),
            digest='755ced02ce7befdb13b7ca74e1e4d08cddba4986afdb63a480f2c93d3140383f',
            size=4683075440,
            details=ModelDetails(
                parent_model='',
                format='gguf',
                family='qwen2',
                families=['qwen2'],
                parameter_size='7.6B',
                quantization_level='Q4_K_M'
            )
        )
    ]
)

Wynik pokazuje modele dostępne lokalnie w Ollama. W tym przypadku mamy pobrany model `deepseek-r1:7b`. Widzimy też podstawowe informacje techniczne: rozmiar pliku, rodzinę modelu, liczbę parametrów oraz poziom kwantyzacji. Oznacza to, że model jest gotowy do użycia w kolejnych komórkach notebooka.

### Tekst wejściowy

To będzie nasz tekst wejściowy dla modelu. W kolejnych krokach poprosimy LLM o analizę tego zgłoszenia i zamianę go na uporządkowany obiekt JSON.

In [7]:
SOURCE_TEXT = """
Klient zgłasza, że po ostatniej aktualizacji aplikacji mobilnej logowanie przez Google działa bardzo wolno.
Czasem pojawia się błąd 504. Problem dotyczy wersji Android 14 i występuje głównie wieczorem.
Klient prosi o pilną reakcję, bo dotyczy to kont firmowych.
""".strip()

print(SOURCE_TEXT)


Klient zgłasza, że po ostatniej aktualizacji aplikacji mobilnej logowanie przez Google działa bardzo wolno.
Czasem pojawia się błąd 504. Problem dotyczy wersji Android 14 i występuje głównie wieczorem.
Klient prosi o pilną reakcję, bo dotyczy to kont firmowych.

# 1. Free-form generation

Najpierw uruchamiamy model bez wymuszania struktury. To jest typowe użycie LLM-a: dajemy instrukcję i oczekujemy odpowiedzi tekstowej. Nie różni się ono niczym od tego co realizowane było na poprzednich zajęciach (z uzupełnieniem szblonu wejściowego).


In [8]:
messages = [
    {
        "role": "user",
        "content": f"""
Przeanalizuj zgłoszenie i zwróć: streszczenie, priorytet, obszar produktu oraz rekomendowane działania.
Nie musisz używać JSON-a.

ZGŁOSZENIE:
{SOURCE_TEXT}
""".strip(),
    }
]

### Odpowiedź modelu

Tak jak było przy tematyce inferencji, modele przeważnie posiadają pewien stopień kreatywności (hiperparametry), dlatego oczekiwany wynik dla 3 generacji będzie dość losowy, lecz będzie zmieżał ku pewnej odpowiedzi.

In [9]:
for i in range(3):
    print(f"\n--- RUN {i + 1} ---")
    print(call_llm(messages, options=STOCHASTIC_OPTIONS))

--- RUN 1 ---

**Streszczenie:**  
Zgłóśniono problem z logowaniem przez Google w aplikacji mobilnej w wersji 14 Android, gdzie czasami pojawia się 
błąd 504. Brzmiande się to jest w kontekście kontów firmowych.

**Prawidłowość:**  
Problem istnieje i jest prioritetu 1.

**Obszar produktu:**  
Odpowiedzialny za zdarzenie jest **system logowania przez Google w wersji 14 Android**.

**Rekomendowane działania:**  
- Zaktualizować aplikację do najnowszej wersji.
- Próba pojawia się błąd 504 jest często spotykana w systemach internetowych, gdy sekcja logowania nie jest 
poprawnie zdefiniowana.

--- RUN 2 ---

**Streszczenie:**  
Klient zasługi logowania przez Google na ANDROeidzie 14 działa wolno, czasami powoduje błąd 504, a jest on 
istniejący w wieczoru.

**Prioirty:**  
1. Logowanie przez Google na ANDROeidzie 14
2. Problem z logowanieą (błąd 504)
3. Wersja ANDROeid 14

**Obszar produktu:**  
Produktem jest aplikacja mobilna Klienta, a konkretnie:  
- Logowanie przez Google
- Wersja ANDROeid 14

**Rekomendowane działania:**  
1. Zaktualizuj wersję ANDROeid do 15.  
2. Sprawdź aktualnościę operacji i aplikacji.  
3. Jeśli problem持续, skontaktuj się z Obsługą Błędu Klienta.

--- RUN 3 ---

**Streszczenie:**  
Zgłoszenie orzeka, że po aktualizacji aplikacji mobilnej logowanie przez Google działa wolno, czasami powstaje błąd
504. Problem koncentruje się na wersji Android 14 i występuje преимуществowo wieczorem.

**Priosztet:**  
Priosztet brzmi istotny, ponieważ klienter operuje kontem firmowym, więc解决问题刻不容缓。

**Obszar produktu:**  
Obszarem produktu jest **logowanie przez Google w aplikacji mobilnej Android 14**.

**Rekomendowane działania:**  
- Zaktualizować sprzęt do najwyższego stanu i zresetować kolejkę naurtów.  
- Wymknąć all-in-one w systemie operacyjnym, a następnie odświeżyć aplikację.  
- Aktualizować aplikację do VERSION 1.0 (Najnowsza wersja).  
- Jeśli problem pozostaje, skontaktować się z Konsultacjami WolframAlpha.

## Problem

Free-form generation może być dobry dla człowieka, ale jest słaby jako element pipeline'u danych, szczególnie przy wdrożeniu takiego modelu w aplikacji/systemach. Do typowych problemów należą:

- zmienne nazwy sekcji,
- różna długość odpowiedzi,
- mieszanie języków,
- dopisywanie informacji spoza źródła,
- brak stabilnego formatu do parsowania.



# 2. JSON mode

W Ollama można przekazać:

```python
format="json"
```

Ten tryb wymusza, żeby odpowiedź modelu była poprawnym JSON-em, ale nie narzuca jeszcze konkretnej struktury. Model sam decyduje, jakie pola umieści w odpowiedzi. Jest to istotny etap pośredni:

- wynik jest łatwiejszy do parsowania niż zwykły tekst,
- można użyć `json.loads(...)`,
- ale nadal nie mamy gwarancji, że pojawią się dokładnie te pola, których oczekuje aplikacja.

To nadal nie jest pełny kontrakt. JSON mode pilnuje formatu JSON, ale nie pilnuje tego, jakie pola i wartości mają się pojawić.


In [10]:
# @title JSON mode
json_mode_messages = [
    {
        "role": "system",
        "content": "Zwracasz wyłącznie poprawny JSON. Bez Markdowna i bez komentarzy.",
    },
    {
        "role": "user",
        "content": f"""
Przeanalizuj zgłoszenie i zwróć JSON z polami, które uznasz za potrzebne.

ZGŁOSZENIE:
{SOURCE_TEXT}
""".strip(),
    },
]

raw_json_mode = call_llm(
    json_mode_messages,
    output_format="json",
    options=DETERMINISTIC_OPTIONS,
    think=False,
)

print(raw_json_mode)

parsed_json_mode = json.loads(raw_json_mode)
print("\nParsed JSON:")
print(parsed_json_mode)


{
  "konsultacje": {
    "opini": [
      {
        "ocena": 5,
        "opini": "Problem logowania przez Google w aplikacji mobilnej jest duży i utrudnia pracę klienta."
      }
    ]
  }
}

Parsed JSON:

{
    'konsultacje': {
        'opini': [
            {
                'ocena': 5,
                'opini': 'Problem logowania przez Google w aplikacji mobilnej jest duży i utrudnia pracę klienta.'
            }
        ]
    }
}

## Ograniczenie JSON mode

JSON mode daje nam składnię JSON, ale nie daje stabilnego modelu danych.

Model może zwrócić:

```json
{"priority": "high"}
```

albo:

```json
{"severity": "urgent"}
```

albo:

```json
{"importance": 4}
```

Wszystkie te odpowiedzi mogą mieć sens, ale aplikacja potrzebuje jednego kontraktu.


# 3. JSON Schema + Pydantic

Teraz definiujemy kontrakt danych po stronie Pythona.

Użyjemy krótkiego schematu. To jest ważne przy lokalnych modelach, szczególnie reasoningowych. Długi schemat i długie pola zwiększają ryzyko ucięcia odpowiedzi.

Kontrakt będzie miał tylko pięć pól:

- `summary`,
- `priority`,
- `product_area`,
- `source_quote`,
- `next_action`.


### Definiowanie wyjścia kontraktu

`TicketAnalysis` to kontrakt danych. Opisuje pola, których oczekujemy od LLM-a, ich typy oraz ograniczenia długości. W Pydantic pracujemy w taki sposób:

1. Tworzymy klasę dziedziczącą po `BaseModel`.
2. Dodajemy pola, np. `summary`, `priority`, `product_area`.
3. Przy polach definiujemy ograniczenia, np. `min_length`, `max_length`.
4. Dla wartości zamkniętych używamy `Literal`, np. tylko `"low"`, `"medium"`, `"high"` albo `"critical"`.
5. Przez `model_json_schema()` zamieniamy model Pydantic na JSON Schema.
6. Ten schemat przekażemy później do Ollama jako `format=schema`.

`ConfigDict(extra="forbid")` oznacza, że model nie może dodać żadnych dodatkowych pól spoza schematu.


In [11]:
class TicketAnalysis(BaseModel):
    model_config = ConfigDict(extra="forbid")

    summary: str = Field(
        min_length=10,
        max_length=120,
        description="Jedno krótkie zdanie streszczające zgłoszenie.",
    )
    priority: Literal["low", "medium", "high", "critical"] = Field(
        description="Priorytet biznesowy zgłoszenia.",
    )
    product_area: Literal["login", "mobile_app", "performance", "other"] = Field(
        description="Najbardziej pasujący obszar produktu.",
    )
    source_quote: str = Field(
        min_length=5,
        max_length=80,
        description="Krótki dosłowny fragment z tekstu źródłowego.",
    )
    next_action: str = Field(
        min_length=5,
        max_length=120,
        description="Jedno krótkie rekomendowane działanie.",
    )


SOURCE_QUOTE_CANDIDATES = [
    "logowanie przez Google",
    "działa bardzo wolno",
    "błąd 504",
    "Android 14",
    "wieczorem",
    "kont firmowych",
]

schema = TicketAnalysis.model_json_schema()
schema["properties"]["source_quote"]["enum"] = SOURCE_QUOTE_CANDIDATES
schema["properties"]["source_quote"]["description"] = "Wybierz dokładnie jedną wartość z enum."

print(json.dumps(schema, ensure_ascii=False, indent=2))


{
  "additionalProperties": false,
  "properties": {
    "summary": {
      "description": "Jedno krótkie zdanie streszczające zgłoszenie.",
      "maxLength": 120,
      "minLength": 10,
      "title": "Summary",
      "type": "string"
    },
    "priority": {
      "description": "Priorytet biznesowy zgłoszenia.",
      "enum": [
        "low",
        "medium",
        "high",
        "critical"
      ],
      "title": "Priority",
      "type": "string"
    },
    "product_area": {
      "description": "Najbardziej pasujący obszar produktu.",
      "enum": [
        "login",
        "mobile_app",
        "performance",
        "other"
      ],
      "title": "Product Area",
      "type": "string"
    },
    "source_quote": {
      "description": "Wybierz dokładnie jedną wartość z enum.",
      "maxLength": 80,
      "minLength": 5,
      "title": "Source Quote",
      "type": "string",
      "enum": [
        "logowanie przez Google",
        "działa bardzo wolno",
        "błąd 504",
        "Android 14",
        "wieczorem",
        "kont firmowych"
      ]
    },
    "next_action": {
      "description": "Jedno krótkie rekomendowane działanie.",
      "maxLength": 120,
      "minLength": 5,
      "title": "Next Action",
      "type": "string"
    }
  },
  "required": [
    "summary",
    "priority",
    "product_area",
    "source_quote",
    "next_action"
  ],
  "title": "TicketAnalysis",
  "type": "object"
}

#### `source_quote` i enum

Model często parafrazuje. Jeżeli poprosimy go o cytat, może wygenerować coś, co brzmi jak cytat, ale nim nie jest.

Dlatego w tym kursie `source_quote` jest wyborem z krótkiej listy fragmentów, które rzeczywiście występują w tekście źródłowym.


### Generowanie z schematem JSON
Model ma zwrócić krótki JSON zgodny z kontraktem `TicketAnalysis`. Następnie Pydantic sprawdza, czy wynik rzeczywiście pasuje do schematu.

In [12]:
STRUCTURED_OPTIONS = {
    **DETERMINISTIC_OPTIONS,
    "temperature": 0,
    "top_p": 1,
    "top_k": 1,
    "seed": 42,
    "num_predict": 2048,
}

schema_messages = [
    {
        "role": "system",
        "content": """
Jesteś ekstraktorem danych z tekstu.
Zwracasz wyłącznie kompletny JSON zgodny ze schematem.
Nie używaj Markdowna.
Nie dodawaj komentarzy.
Nie dodawaj pól spoza schematu.
Pisz krótko i po polsku.
source_quote wybierz dokładnie z enum.
""".strip(),
    },
    {
        "role": "user",
        "content": f"""
Wypełnij JSON zgodnie ze schematem.

SCHEMAT JSON:
{json.dumps(schema, ensure_ascii=False, indent=2)}

TEKST ŹRÓDŁOWY:
{SOURCE_TEXT}
""".strip(),
    },
]

raw_structured = call_llm(
    schema_messages,
    output_format=schema,
    options=STRUCTURED_OPTIONS,
    think=False,
)

print("Raw model output:")
print(raw_structured)

analysis = TicketAnalysis.model_validate_json(raw_structured)

print("\nValidated Pydantic object:")
print(analysis)


Raw model output:

{
  "summary": "Klient zasługi logowania przez Google i błędu 504 w wersji Android 14, który występuje wieczorem i 
dotyczy kontu firmęck"
  ,
  "priority": "critical",
  "product_area": "mobile_app",
  "source_quote": "logowanie przez Google",
  "next_action": "Sprawdź wersję operacji mobilnej i zaktualizuj ją."
}

Validated Pydantic object:

TicketAnalysis(
    summary='Klient zasługi logowania przez Google i błędu 504 w wersji Android 14, który występuje wieczorem i 
dotyczy kontu firmęck',
    priority='critical',
    product_area='mobile_app',
    source_quote='logowanie przez Google',
    next_action='Sprawdź wersję operacji mobilnej i zaktualizuj ją.'
)

# 4. Walidacja semantyczna

Pydantic sprawdza kształt danych:

- typy,
- wymagane pola,
- długości tekstu,
- wartości enum,
- brak dodatkowych pól.

Nie sprawdza jednak całej semantyki.

Przykład: `source_quote` powinien występować w tekście źródłowym. To sprawdzimy osobną funkcją.


In [13]:
def normalize_text(text: str) -> str:
    """
    Normalizuje tekst przed porównaniem.

    Zamieniamy tekst na małe litery i usuwamy nadmiarowe spacje,
    żeby porównanie było odporne na różnice w wielkości liter i formatowaniu.
    """
    return " ".join(str(text).lower().split())


def validate_or_repair_source_quote(obj: TicketAnalysis, source_text: str) -> TicketAnalysis:
    """
    Sprawdza, czy source_quote faktycznie występuje w tekście źródłowym.

    Jeśli source_quote jest poprawnym cytatem, zwracamy obiekt bez zmian.
    Jeśli model zwrócił parafrazę albo tekst spoza źródła, naprawiamy pole,
    wybierając pierwszy kandydat z SOURCE_QUOTE_CANDIDATES, który występuje w source_text.
    """
    source_norm = normalize_text(source_text)
    quote_norm = normalize_text(obj.source_quote)

    # Przypadek poprawny: cytat z modelu znajduje się dosłownie w tekście źródłowym.
    if quote_norm in source_norm:
        print("source_quote OK:", obj.source_quote)
        return obj

    # Przypadek błędny: model zwrócił parafrazę albo nieistniejący fragment.
    print("source_quote repaired:")
    print("before:", obj.source_quote)

    # Szukamy bezpiecznej wartości zastępczej w przygotowanej liście cytatów.
    for candidate in SOURCE_QUOTE_CANDIDATES:
        if normalize_text(candidate) in source_norm:
            repaired = obj.model_copy(update={"source_quote": candidate})
            print("after:", repaired.source_quote)
            return repaired

    # Jeśli nie znaleziono żadnego poprawnego kandydata, zostawiamy obiekt bez zmian.
    print("No repair candidate found. Object left unchanged.")
    return obj


analysis = validate_or_repair_source_quote(analysis, SOURCE_TEXT)
analysis

source_quote OK: logowanie przez Google

TicketAnalysis(summary='Klient zasługi logowania przez Google i błędu 504 w wersji Android 14, który występuje wieczorem i dotyczy kontu firmęck', priority='critical', product_area='mobile_app', source_quote='logowanie przez Google', next_action='Sprawdź wersję operacji mobilnej i zaktualizuj ją.')

### Przykład błedu/parafrazy

In [14]:
bad_example = TicketAnalysis(
    summary="Klient ma problem z logowaniem po aktualizacji aplikacji.",
    priority="high",
    product_area="login",
    source_quote="Ten fragment nie występuje w tekście źródłowym",
    next_action="Sprawdzić logowanie Google na Android 14.",
)

print("Before repair:")
print(bad_example)

fixed_example = validate_or_repair_source_quote(bad_example, SOURCE_TEXT)

print("\nAfter repair:")
print(fixed_example)


Before repair:

TicketAnalysis(
    summary='Klient ma problem z logowaniem po aktualizacji aplikacji.',
    priority='high',
    product_area='login',
    source_quote='Ten fragment nie występuje w tekście źródłowym',
    next_action='Sprawdzić logowanie Google na Android 14.'
)

source_quote repaired:

before: Ten fragment nie występuje w tekście źródłowym

after: logowanie przez Google

After repair:

TicketAnalysis(
    summary='Klient ma problem z logowaniem po aktualizacji aplikacji.',
    priority='high',
    product_area='login',
    source_quote='logowanie przez Google',
    next_action='Sprawdzić logowanie Google na Android 14.'
)

# 5. Retry po błędzie walidacji

W praktyce pojedyncze wywołanie modelu nie wystarcza.

Poprawny przepływ wygląda tak:

1. generujemy JSON,
2. parsujemy przez Pydantic,
3. wykonujemy walidację semantyczną,
4. jeśli JSON jest błędny, prosimy model o poprawkę,
5. ograniczamy liczbę prób.

W tej wersji błędy semantyczne pola `source_quote` są naprawiane lokalnie, bo mamy kontrolowaną listę poprawnych cytatów.


In [15]:
def build_ticket_messages(source_text: str, schema: dict) -> list[dict[str, str]]:
    return [
        {
            "role": "system",
            "content": (
                "Jesteś deterministycznym ekstraktorem danych. "
                "Zwracasz wyłącznie mały JSON zgodny ze schematem. "
                "Nie dodajesz Markdowna ani tekstu poza JSON-em. "
                "Nie dodajesz pól spoza schematu. "
                "source_quote wybierz dokładnie z enum. "
                "Pisz krótko po polsku."
            ),
        },
        {
            "role": "user",
            "content": f"""
Wypełnij JSON zgodnie ze schematem.

SCHEMAT JSON:
{json.dumps(schema, ensure_ascii=False, indent=2)}

TEKST ŹRÓDŁOWY:
{source_text}

Zwróć wyłącznie JSON.
""".strip(),
        },
    ]


def generate_ticket_analysis(source_text: str, max_attempts: int = 3) -> tuple[TicketAnalysis, str, int]:
    messages = build_ticket_messages(source_text, schema)
    last_error = None
    last_raw = None

    for attempt in range(1, max_attempts + 1):
        raw = call_llm(
            messages,
            output_format=schema,
            options=STRUCTURED_OPTIONS,
            think=False,
        ).strip()

        last_raw = raw

        try:
            obj = TicketAnalysis.model_validate_json(raw)
            obj = validate_or_repair_source_quote(obj, source_text)
            return obj, raw, attempt

        except ValidationError as e:
            last_error = str(e)
            messages = build_ticket_messages(source_text, schema)
            messages.append(
                {
                    "role": "user",
                    "content": (
                        "Poprzednia odpowiedź była błędna. "
                        "Zwróć ponownie wyłącznie kompletny JSON zgodny ze schematem. "
                        "Nie dodawaj pól spoza schematu. "
                        "source_quote wybierz dokładnie z enum. "
                        f"Błąd: {last_error}"
                    ),
                }
            )

    raise RuntimeError(
        f"Nie udało się uzyskać poprawnego wyniku po {max_attempts} próbach.\n"
        f"Ostatni błąd: {last_error}\n"
        f"Ostatnia odpowiedź: {last_raw}"
    )


obj, raw, attempts = generate_ticket_analysis(SOURCE_TEXT)

print("Attempts:", attempts)
print(obj)


source_quote OK: błąd 504

Attempts: 1

TicketAnalysis(
    summary='Klient zasługi logowania przez Google i błędu 504 w wersji Android 14, który występuje wieczorem i 
dotyczy kontu firmęck',
    priority='critical',
    product_area='mobile_app',
    source_quote='błąd 504',
    next_action='Sprawdź aktualizację aplikacji mobilnej i skontaktuj się z naszym supportem.'
)

# 6. Kanonizacja JSON i test regresji

Jeżeli wynik modelu ma wejść do aplikacji, dobrze jest mieć prosty test powtarzalności.

Nie porównujemy surowego tekstu, bo wcięcia i kolejność kluczy mogą się różnić.

Porównujemy kanoniczny JSON:

- stała kolejność kluczy,
- brak zbędnych spacji,
- hash SHA-256.


In [16]:
runs = []

for i in range(3):
    obj, raw, attempts = generate_ticket_analysis(SOURCE_TEXT)
    canonical = canonical_json(obj)
    runs.append(
        {
            "run": i + 1,
            "attempts": attempts,
            "canonical": canonical,
            "hash": sha256_short(canonical),
        }
    )

for item in runs:
    print(f"RUN {item['run']} | attempts={item['attempts']} | hash={item['hash']}")
    print(item["canonical"])
    print()

unique_hashes = sorted({item["hash"] for item in runs})
print("Unique hashes:", unique_hashes)


source_quote OK: błąd 504

source_quote OK: błąd 504

source_quote OK: błąd 504

RUN 1 | attempts=1 | hash=c6b90dc851c2

{"next_action":"Sprawdź wersję operacji systemu i zaktualizuj 
aplikację.","priority":"critical","product_area":"mobile_app","source_quote":"błąd 504","summary":"Klient zasługi 
logowania przez Google i błędu 504 w wersji Android 14, który występuje wieczorem i dotyczy kontu firmęck"}

RUN 2 | attempts=1 | hash=c6b90dc851c2

{"next_action":"Sprawdź wersję operacji systemu i zaktualizuj 
aplikację.","priority":"critical","product_area":"mobile_app","source_quote":"błąd 504","summary":"Klient zasługi 
logowania przez Google i błędu 504 w wersji Android 14, który występuje wieczorem i dotyczy kontu firmęck"}

RUN 3 | attempts=1 | hash=c6b90dc851c2

{"next_action":"Sprawdź wersję operacji systemu i zaktualizuj 
aplikację.","priority":"critical","product_area":"mobile_app","source_quote":"błąd 504","summary":"Klient zasługi 
logowania przez Google i błędu 504 w wersji Android 14, który występuje wieczorem i dotyczy kontu firmęck"}

Unique hashes:
['c6b90dc851c2']

Jeżeli hashe są identyczne, pipeline zachował się powtarzalnie w tym środowisku.

Jeżeli hashe są różne, trzeba sprawdzić:

- ustawienia samplingowe,
- długość promptu,
- długość schematu,
- wersję modelu,
- wersję Ollama,
- backend GPU/CPU.


# 7. Porównanie profili generacji

Teraz porównamy trzy profile:

- losowy,
- `temperature=0` z większym `top_k`,
- najbardziej zachowawczy `temperature=0`, `top_k=1`.

Wynik dalej przechodzi przez ten sam kontrakt i tę samą walidację.


In [17]:
OPTION_PROFILES = {
    "stochastic": {
        "temperature": 0.6,
        "top_k": 40,
        "top_p": 0.9,
        "num_predict": 2048,
        "num_ctx": 4096,
    },
    "temp0_topk40": {
        "temperature": 0,
        "seed": 42,
        "top_k": 40,
        "top_p": 1,
        "num_predict": 2048,
        "num_ctx": 4096,
    },
    "temp0_topk1": {
        "temperature": 0,
        "seed": 42,
        "top_k": 1,
        "top_p": 1,
        "num_predict": 2048,
        "num_ctx": 4096,
    },
}


def generate_ticket_with_options(source_text: str, options: dict) -> TicketAnalysis:
    messages = build_ticket_messages(source_text, schema)
    raw = call_llm(messages, output_format=schema, options=options, think=False)
    obj = TicketAnalysis.model_validate_json(raw)
    obj = validate_or_repair_source_quote(obj, source_text)
    return obj


for profile_name, options in OPTION_PROFILES.items():
    hashes = []

    print(f"\n=== {profile_name} ===")

    for i in range(3):
        try:
            obj = generate_ticket_with_options(SOURCE_TEXT, options)
            h = sha256_short(canonical_json(obj))
            hashes.append(h)
            print(f"run={i + 1}, hash={h}")
        except Exception as e:
            print(f"run={i + 1}, validation_failed={type(e).__name__}: {e}")

    print("unique hashes:", sorted(set(hashes)))


=== stochastic ===

source_quote OK: błąd 504

run=1, hash=7966ce779df9

source_quote OK: błąd 504

run=2, hash=3045a7e1a76c

...

source_quote OK: błąd 504

run=2, hash=f1acdd4c3118

source_quote OK: błąd 504

run=3, hash=f1acdd4c3118

unique hashes:
['f1acdd4c3118']

# 8. Drugi kontrakt: routing zgłoszenia

Structured generation nie musi oznaczać tylko ekstrakcji danych.

Można też wymusić decyzję biznesową w kontrolowanym formacie. Przykład: routing zgłoszenia do zespołu.


In [18]:
class TicketRouting(BaseModel):
    model_config = ConfigDict(extra="forbid")

    team: Literal["mobile", "backend", "security", "customer_success"] = Field(
        description="Zespół, do którego powinno trafić zgłoszenie."
    )
    sla: Literal["standard", "urgent", "immediate"] = Field(
        description="Poziom reakcji SLA."
    )
    reason: str = Field(
        min_length=10,
        max_length=140,
        description="Krótkie uzasadnienie decyzji."
    )
    needs_manager_escalation: bool = Field(
        description="Czy sprawa wymaga eskalacji do managera."
    )


routing_schema = TicketRouting.model_json_schema()
print(json.dumps(routing_schema, ensure_ascii=False, indent=2))


{
  "additionalProperties": false,
  "properties": {
    "team": {
      "description": "Zespół, do którego powinno trafić zgłoszenie.",
      "enum": [
        "mobile",
        "backend",
        "security",
        "customer_success"
      ],
      "title": "Team",
      "type": "string"
    },
    "sla": {
      "description": "Poziom reakcji SLA.",
      "enum": [
        "standard",
        "urgent",
        "immediate"
      ],
      "title": "Sla",
      "type": "string"
    },
    "reason": {
      "description": "Krótkie uzasadnienie decyzji.",
      "maxLength": 140,
      "minLength": 10,
      "title": "Reason",
      "type": "string"
    },
    "needs_manager_escalation": {
      "description": "Czy sprawa wymaga eskalacji do managera.",
      "title": "Needs Manager Escalation",
      "type": "boolean"
    }
  },
  "required": [
    "team",
    "sla",
    "reason",
    "needs_manager_escalation"
  ],
  "title": "TicketRouting",
  "type": "object"
}

In [19]:
routing_messages = [
    {
        "role": "system",
        "content": (
            "Klasyfikujesz zgłoszenia do zespołów. "
            "Zwracasz wyłącznie mały JSON zgodny ze schematem."
        ),
    },
    {
        "role": "user",
        "content": f"""
Zdecyduj, do którego zespołu powinno trafić zgłoszenie.

SCHEMAT JSON:
{json.dumps(routing_schema, ensure_ascii=False, indent=2)}

ZGŁOSZENIE:
{SOURCE_TEXT}
""".strip(),
    },
]

raw_routing = call_llm(
    routing_messages,
    output_format=routing_schema,
    options=STRUCTURED_OPTIONS,
    think=False,
)

routing = TicketRouting.model_validate_json(raw_routing)

print(routing)
print("Canonical hash:", sha256_short(canonical_json(routing)))


TicketRouting(
    team='mobile',
    sla='urgent',
    reason='Klient zgłasza, że po ostatniej aktualizacji aplikacji mobilnej logowanie przez Google działa bardzo 
wolno. Czasem pojawia się błąd 504. Brz',
    needs_manager_escalation=True
)

Canonical hash: af304698261d

# 9. Własny wariant modelu w Ollama

Jeżeli ten sam model ma być często używany do structured generation, można stworzyć wariant z domyślnymi parametrami.

To nie zastępuje walidacji, ale zmniejsza ryzyko przypadkowego uruchomienia z innymi ustawieniami.


In [20]:
CREATE_DETERMINISTIC_MODEL = False # @param {type:"boolean"}
DETERMINISTIC_MODEL_NAME = "structured-local" # @param {type:"string"}

modelfile = f"""
FROM {model}

PARAMETER temperature 0
PARAMETER top_k 1
PARAMETER top_p 1
PARAMETER seed 42
PARAMETER num_predict 2048

SYSTEM Jesteś deterministycznym ekstraktorem danych. Zwracasz wyłącznie mały JSON zgodny z przekazanym schematem.
""".strip()

with open("Modelfile.structured", "w", encoding="utf-8") as f:
    f.write(modelfile)

print(modelfile)

if CREATE_DETERMINISTIC_MODEL:
    subprocess.run(
        ["ollama", "create", DETERMINISTIC_MODEL_NAME, "-f", "Modelfile.structured"],
        check=True,
    )
    print("Created:", DETERMINISTIC_MODEL_NAME)
else:
    print("Set CREATE_DETERMINISTIC_MODEL=True to create the model variant.")


FROM deepseek-r1:7b

PARAMETER temperature 0
PARAMETER top_k 1
PARAMETER top_p 1
PARAMETER seed 42
PARAMETER num_predict 2048

SYSTEM Jesteś deterministycznym ekstraktorem danych. Zwracasz wyłącznie mały JSON zgodny z przekazanym schematem.

Set CREATE_DETERMINISTIC_MODEL=True to create the model variant.

# 10. Checklist

Praktyczny przepis na structured generation z lokalnym LLM-em:

1. Użyj konkretnego modelu i konkretnego runtime'u.
2. Ustaw `temperature=0`, `top_k=1`, stały `seed`.
3. Użyj krótkiego schematu JSON.
4. Przekaż JSON Schema do `format`.
5. Powtórz najważniejsze wymagania w promptcie.
6. Waliduj wynik po stronie aplikacji.
7. Dodaj walidację semantyczną tam, gdzie sam schemat nie wystarcza.
8. Dodaj retry dla błędów składniowych i walidacyjnych.
9. Kanonizuj JSON.
10. Testuj powtarzalność przez hashe.

Najważniejsza zasada: model generuje kandydat na dane, ale to aplikacja decyduje, czy wynik jest poprawny.


# 11. Ćwiczenia

## Ćwiczenie 1

Zmień `SOURCE_TEXT` na inne zgłoszenie i uruchom sekcje od definicji kontraktu.

Sprawdź, czy:

- model zwraca poprawny JSON,
- `source_quote` jest poprawnym fragmentem tekstu,
- hashe są powtarzalne.

## Ćwiczenie 2

Dodaj pole:

```python
customer_impact: Literal["single_user", "many_users", "enterprise_accounts", "unknown"]
```

Następnie wygeneruj schemat i sprawdź, czy lokalny model poprawnie wypełnia nowe pole.

## A. Ćwiczenie 3

Zbuduj własny extractor dla jednego z przypadków:

- faktura,
- CV,
- opinia klienta,
- log systemowy,
- wiadomość e-mail.

Wymagania:

1. Klasa Pydantic.
2. JSON Schema.
3. Wywołanie przez Ollama.
4. Walidacja Pydantic.
5. Prosta walidacja semantyczna.
6. Test 3 powtórzeń i porównanie hashy.

## B. Ćwiczenie 3

Zbuduj własny extractor dla swojego projektu.


In [21]:
class MyExtractorOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    title: str = Field(min_length=3, max_length=100)
    category: Literal["a", "b", "c"]
    source_quote: str = Field(min_length=3, max_length=80)


MY_TEXT = """
Wklej tutaj własny tekst wejściowy.
""".strip()

my_schema = MyExtractorOutput.model_json_schema()

my_messages = [
    {
        "role": "system",
        "content": "Zwracasz wyłącznie mały JSON zgodny ze schematem. Bez Markdowna.",
    },
    {
        "role": "user",
        "content": f"""
SCHEMAT JSON:
{json.dumps(my_schema, ensure_ascii=False, indent=2)}

TEKST:
{MY_TEXT}
""".strip(),
    },
]

# raw = call_llm(my_messages, output_format=my_schema, options=STRUCTURED_OPTIONS, think=False)
# obj = MyExtractorOutput.model_validate_json(raw)
# print(obj)
# print(canonical_json(obj))


## Uzupełnienie

W tym notebooku korzystamy z Ollama, ponieważ daje prosty lokalny mechanizm structured output przez `format=schema`.

Warto jednak wiedzieć, że podobne podejście istnieje także w ekosystemie Hugging Face. Nie dotyczy to bezpośrednio samego `from_pretrained(...)`, ale warstwy inferencyjnej, np. Hugging Face Inference Providers, gdzie można przekazać schemat odpowiedzi przez `response_format`.

W celu uzupełnienia tematu sprawdźcie dokumentację:

https://huggingface.co/docs/inference-providers/guides/structured-output